In [18]:
from pathlib import Path
import pandas as pd

## Unir todas as bases de dados

In [11]:
pasta = Path("dados-brutos")

arquivos = sorted(pasta.glob("atendimentos-*.csv"))

dfs = []

for arquivo in arquivos:
    df = pd.read_csv(arquivo, encoding="utf-8", sep=';')
    dfs.append(df)

dados = pd.concat(dfs, ignore_index=True)

print(f"Total de registros: {len(dados):,}")

Total de registros: 886,264


In [12]:
dados.head()

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,Sul,2014,Abril,2014-04-01,NaN,NaN,"2ª Travessa Soldado Virgilio Lucio, %$<!",Cohab,Ur 04,R1,2014-04-01,Acoes de Porta-Porta,NaN,NaN,NaN,0.0
1,Sul,2014,Abril,2014-04-16,NaN,Vistoria,"Rua Taipu, #_|/",Cohab,Tres Carneiros Alto,R2,2014-04-16,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
2,Sul,2014,Abril,2014-04-16,NaN,Vistoria,"Rua Taipu, .[?",Cohab,Tres Carneiros Alto,R2,2014-04-16,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
3,Sul,2014,Abril,2014-04-16,NaN,Vistoria,"Rua Paipu, #$[",Cohab,Tres Carneiros Alto,R2,2014-04-16,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
4,Sul,2014,Abril,2014-04-16,NaN,Vistoria,"Rua Ibirapua, `=-[",Cohab,Tres Carneiros Alto,R2,2014-04-16,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0


In [13]:
dados.describe()

,Quantidade,Altura,Largura,Area_m
count,136695.000000,130188.000000,130203.000000,886263.000000
mean,1.829811,17.984230,10.157354,39.213176
std,1.409308,10.125517,27.704759,114.626113
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,15.000000,8.000000,0.000000
50%,1.000000,15.000000,8.000000,0.000000
75%,2.000000,20.000000,8.000000,0.000000
max,70.000000,784.000000,1480.000000,4920.000000


## Tipos de ocorrências existentes

In [15]:
ocorrencias = (dados["Ocorrencia"].fillna("").str.strip().value_counts())
ocorrencias.to_csv("dados-tratados/tipos_ocorrencia.csv", encoding="utf-8-sig")
ocorrencias.head(100)

Ocorrencia
                                    610462
Monitoramento                       163347
Colocao de Lonas Plsticas            50902
Vistoria                             20701
Imóveis com Danos / Risco            16746
                                     ...  
Entrega Donativos                        1
Monitoramento Produtos Perigosos         1
COLOcao de Lonas Plsticas                1
vistoria em Imveis                       1
Fechamento de Obras                      1
Name: count, Length: 72, dtype: int64

In [16]:
ocorrencias = (dados["Solicitacao"].fillna("").str.strip().value_counts())
ocorrencias.to_csv("dados-tratados/tipos_solicitacao.csv", encoding="utf-8-sig")
ocorrencias.head(100)

Solicitacao
Monitoramento                   281968
                                135385
Colocação de Lonas Plásticas     66797
Vistoria                         42912
Vistoria em Imóveis              13873
                                 ...  
"Rua A                             504
"Avenida Melvin Jones              499
"Travessa Guaraciaba               496
"Rua Aguas Claras                  482
"Rua Leoncio Rodrigues             481
Name: count, Length: 100, dtype: int64

## Filtrar registros de alagamento

In [20]:
keywords = 'alagado|alagamento'

filtro_ocorrencia = (
    dados["Ocorrencia"]
    .fillna("")
    .str.lower()
    .str.contains(keywords, regex=True)
)

filtro_solicitacao = (
    dados["Solicitacao"]
    .fillna("")
    .str.lower()
    .str.contains(keywords, regex=True)
)

alagamentos = dados[filtro_ocorrencia | filtro_solicitacao].copy()
print(alagamentos.shape)

(4284, 16)


In [21]:
alagamentos.head()

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
643,Planície,2014,Abril,NaN,Imóveis Alagados,Monitoramento Alagado,"Rua Rio do Prata, @}>",Ibura de Baixo,Nao Informada,R1,2014-04-15,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
644,Planície,2014,Abril,NaN,Imóveis Alagados,Monitoramento Alagado,"Rua Flores da Cunha, 9<-%",Ibura de Baixo,Nao Informada,R1,2014-04-15,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
645,Planície,2014,Abril,NaN,Imóveis Alagados,Monitoramento Alagado,"Rua Santa Leopoldina, %$<!",Ibura de Baixo,Nao Informada,R1,2014-04-15,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
657,Planície,2014,Abril,NaN,Imóveis Alagados,Monitoramento Alagado,"Avenida Dois Rios, .<^&",Ibura de Baixo,Nao Informada,R1,2014-04-15,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
668,Planície,2014,Abril,NaN,Imóveis Alagados,Monitoramento Alagado,"Avenida Dois Rios, ^[{",Ibura de Baixo,Nao Informada,R1,2014-04-15,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0


In [22]:
alagamentos["Ocorrencia"].value_counts()

Ocorrencia
Monitoramento Alagado          867
Alagamentos                    497
Vistoria Setor Alagado         484
Imóveis Alagados               284
Imóveis com Danos / Risco      222
Vistoria Alagado               146
Imóveis com danos / Risco       20
Monitoramento Setor Alagado     15
Incêndios                        1
Name: count, dtype: int64

In [23]:
alagamentos["Solicitacao"].value_counts()

Solicitacao
Monitoramento Alagado                      2056
Vistoria Alagado                            396
"Avenida Melvin Jones                       242
Vistoria Setor Alagado                      139
"Rua Rio Morno                              134
                                           ... 
"Rio Moxoto                                   1
"Rua Herval                                   1
"Rua Expedicionario Francisco Vitoriano       1
"Comumidade Sertanopolis                      1
"Rua Presidente Venceslau                     1
Name: count, Length: 356, dtype: int64

In [26]:
alagamentos.loc[alagamentos["Ocorrencia"].str.contains("Incêndios", na=False)]

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
25059,Planície,2014,Janeiro,2014-01-16,Incêndios,Vistoria Alagado,"Travessa Ubiratan, ""#*9",Campina do Barreto,Nao Informada,R1,2014-01-16,Analise Visual Estrutural,NaN,NaN,NaN,0.0


In [28]:
alagamentos.loc[alagamentos["Ocorrencia"].str.contains("Imóveis com Danos / Risco", na=False)].head()

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
8349,Planície,2014,Agosto,2014-08-07,Imóveis com Danos / Risco,Monitoramento Alagado,"Estrada do Frigorifico, ^[{",Imbiribeira,Nao Informada,R1,2014-08-07,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
8597,Planície,2014,Agosto,2014-08-07,Imóveis com Danos / Risco,Monitoramento Alagado,"Estrada do Frigorifico, {})!",Imbiribeira,Nao Informada,R1,2014-08-07,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
8598,Planície,2014,Agosto,2014-08-07,Imóveis com Danos / Risco,Monitoramento Alagado,"Estrada do Frigorifico, #^>",Imbiribeira,Nao Informada,R1,2014-08-07,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
8599,Planície,2014,Agosto,2014-08-07,Imóveis com Danos / Risco,Monitoramento Alagado,"Estrada do Frigorifico, ~|}[",Imbiribeira,Nao Informada,R1,2014-08-07,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0
8935,Planície,2014,Agosto,2014-08-07,Imóveis com Danos / Risco,Monitoramento Alagado,"Rua Cafezolpolis, #""#~",Imbiribeira,Nao Informada,R1,2014-08-07,Mapeamento de Area de Risco,NaN,NaN,NaN,0.0


In [32]:
alagamentos.isnull().sum()

Regional            0
Ano                 0
Mês                 0
Data              703
Ocorrencia       1748
Solicitacao         1
Endereco            0
Bairro              1
Localidade          0
Grau_de_Risco    2613
Data_da_Acao        0
Tipo_da_Acao       39
Quantidade       4284
Altura           4284
Largura          4284
Area_m              0
dtype: int64

In [31]:
alagamentos.to_csv("dados-tratados/registros_alagamentos.csv", encoding="utf-8-sig", index=False, sep=';')